# 002 · Generics

> **Syntax › Type Hints** · Python 3.14

Generics let a function or class work with "some type `T`" while the type checker keeps track of which `T` it is. Since Python 3.12 (PEP 695) the syntax looks remarkably like C#: `def first[T](xs: list[T]) -> T` and `class Box[T]:`.

**In this notebook you will learn:**

- generic functions and classes with the modern PEP 695 syntax
- bounds (`[T: Comparable]`) vs C#'s `where T : IComparable<T>`, and constrained type variables
- the older `TypeVar` / `Generic` form you will meet in existing code
- variance basics: inferred in PEP 695 vs C#'s `in` / `out`
- generic `Protocol`s — the closest thing to generic interfaces
- why Python generics are erased at runtime, unlike .NET's reified generics

## C# mental model

| C# | Python 3.12+ |
|---|---|
| `T First<T>(List<T> xs)` | `def first[T](xs: list[T]) -> T:` |
| `class Box<T>` | `class Box[T]:` |
| `where T : IComparable<T>` | `[T: Comparable]` (an upper **bound**) |
| — | `[T: (int, str)]` (T must be exactly one of these) |
| `interface IRepository<T>` | `class Repository[T](Protocol):` |
| `out T` / `in T` (interfaces & delegates only) | variance **inferred** automatically |
| `typeof(T)` inside a generic method | not available — generics are erased at runtime |

The key mental shift: as with all type hints *(see Type Hints › 001 · Type Annotations)*, generics exist for the **type checker**. At runtime `Box[int]` and `Box[str]` produce the same kind of object.

## Generic functions

```csharp
// C#
static T First<T>(IReadOnlyList<T> items) => items[0];

int n = First(new List<int> { 1, 2, 3 });      // T inferred as int
string s = First(new[] { "a", "b" });          // T inferred as string
```

Type parameters go in square brackets after the function name. The checker infers `T` from the arguments, just like C#.

In [ ]:
from collections.abc import Sequence


def first[T](items: Sequence[T]) -> T:
    return items[0]


n = first([1, 2, 3])          # checker infers T = int, so n: int
s = first(("a", "b"))         # T = str
print(n, s)
print(first.__type_params__)  # the type parameters exist at runtime as objects

Multiple type parameters work the same way:

In [ ]:
def swap[A, B](pair: tuple[A, B]) -> tuple[B, A]:      # C#: (B, A) Swap<A, B>((A, B) pair)
    a, b = pair
    return b, a


def to_dict[K, V](keys: Sequence[K], values: Sequence[V]) -> dict[K, V]:
    return dict(zip(keys, values))


print(swap((1, "one")))
print(to_dict(["a", "b"], [1, 2]))

## Generic classes

```csharp
// C#
class Box<T>(T value)
{
    public T Value { get; } = value;
    public Box<TResult> Map<TResult>(Func<T, TResult> f) => new(f(Value));
}
```

In [ ]:
from collections.abc import Callable


class Box[T]:
    def __init__(self, value: T) -> None:
        self.value = value

    def map[R](self, f: Callable[[T], R]) -> Box[R]:   # method with its own type parameter
        return Box(f(self.value))

    def __repr__(self) -> str:
        return f"Box({self.value!r})"


b = Box(21)                    # inferred as Box[int]
print(b.map(lambda x: x * 2))
print(b.map(str).map(len))     # Box[int] -> Box[str] -> Box[int]

You can also spell out the type argument explicitly, like `new Box<int>(21)` — though usually inference is enough:

In [ ]:
explicit = Box[int](21)
print(explicit)

## Bounds: `[T: Bound]` vs `where T : ...`

A **bound** says "`T` can be any type that is a subtype of `Bound`". Since Python uses duck typing, the bound is usually a `Protocol` *(see Classes & OOP › 006 · Abstract Classes and Protocols)* describing the methods `T` must have.

```csharp
// C#
static T Largest<T>(IEnumerable<T> items) where T : IComparable<T>
{
    var best = items.First();
    foreach (var x in items)
        if (x.CompareTo(best) > 0) best = x;
    return best;
}
```

In [ ]:
from typing import Any, Protocol


class Comparable(Protocol):
    def __lt__(self, other: Any, /) -> bool: ...


def largest[T: Comparable](items: Sequence[T]) -> T:
    best = items[0]
    for x in items:
        if best < x:          # allowed: the bound guarantees __lt__
            best = x
    return best


print(largest([3, 9, 4]))                 # int has __lt__
print(largest(["pear", "apple", "zebra"]))   # so does str

A bound can also be a regular class — `[T: Animal]` ≈ `where T : Animal`. The important part is that the return type stays the **specific** `T`, not the bound:

In [ ]:
class Animal:
    def __init__(self, name: str) -> None:
        self.name = name


class Dog(Animal):
    def bark(self) -> str:
        return f"{self.name}: woof"


def rename[T: Animal](animal: T, name: str) -> T:   # returns Dog for a Dog, not just Animal
    animal.name = name
    return animal


print(rename(Dog("rex"), "max").bark())   # the checker knows the result is a Dog

## Constrained type variables: `[T: (A, B)]`

A tuple after the colon means "`T` must be **exactly one of** these types". This has no direct C# equivalent. It's typically used when a function works for a fixed set of unrelated types and must return the same one it received.

In [ ]:
def concat[T: (str, bytes)](a: T, b: T) -> T:
    return a + b


print(concat("ab", "cd"))      # T = str
print(concat(b"ab", b"cd"))    # T = bytes
# concat("ab", b"cd")          # checker error: str and bytes can't both be T

⚠️ **Gotcha:** `[T: (int, str)]` is **not** the same as `[T: int | str]`. With the constraint, both arguments must be the *same* type and the result is that type. With the union bound, `T` could be solved as `int | str`, allowing mixed arguments.

## The older `TypeVar` / `Generic` form

Code written before Python 3.12 (and many libraries today) declares type variables explicitly. You need to be able to **read** it:

```python
from typing import Generic, TypeVar

T = TypeVar("T")                         # plain type variable
N = TypeVar("N", bound=Comparable)       # bounded    ≈ [N: Comparable]
S = TypeVar("S", str, bytes)             # constrained ≈ [S: (str, bytes)]

def first(items: Sequence[T]) -> T: ...  # ≈ def first[T](...)

class Box(Generic[T]):                   # ≈ class Box[T]:
    def __init__(self, value: T) -> None:
        self.value = value
```

In [ ]:
from typing import Generic, TypeVar

T = TypeVar("T")


class OldBox(Generic[T]):
    def __init__(self, value: T) -> None:
        self.value = value


print(OldBox[int](5).value, OldBox.__parameters__)
print(Box.__type_params__)   # the PEP 695 class exposes the same information

💡 For new code targeting 3.12+, prefer the PEP 695 syntax: type parameters are scoped to the function/class (no module-level `T` floating around) and variance is inferred.

## Variance basics

Variance answers: if `Dog` is a subtype of `Animal`, is `Container[Dog]` a subtype of `Container[Animal]`?

| Variance | C# | Python | Example |
|---|---|---|---|
| Covariant | `out T` — `IEnumerable<out T>` | inferred when `T` is only **returned** | `Sequence[Dog]` → `Sequence[Animal]` ✅ |
| Contravariant | `in T` — `Action<in T>` | inferred when `T` is only **accepted** | `Callable[[Animal], None]` → `Callable[[Dog], None]` ✅ |
| Invariant | default — `List<T>` | when `T` is both read and written | `list[Dog]` → `list[Animal]` ❌ |

Two differences from C#:

- C# only allows variance on **interfaces and delegates**, declared with `in`/`out`. With PEP 695, the checker **infers** variance for any generic class from how `T` is used.
- In the old `TypeVar` style you declare it: `TypeVar("T_co", covariant=True)` / `TypeVar("T_contra", contravariant=True)`.

In [ ]:
class Source[T]:                  # T only appears in return positions -> inferred covariant
    def __init__(self, item: T) -> None:
        self._item = item

    def get(self) -> T:
        return self._item


def describe(src: Source[Animal]) -> str:
    return src.get().name


print(describe(Source(Dog("rex"))))   # OK for the checker: Source[Dog] is a Source[Animal]

T_src = Source.__type_params__[0]
print(T_src.__infer_variance__)       # True — PEP 695 type params infer their variance

⚠️ **Gotcha:** just like `List<Dog>` is not a `List<Animal>` in C#, `list[Dog]` is not a `list[Animal]` — a function could append a `Cat` to it. Accept `Sequence[Animal]` (read-only, covariant) when you only need to read.

## Generic Protocols ≈ generic interfaces

A generic `Protocol` describes a shape with a type parameter. Classes match it **structurally** — they don't have to inherit from it.

```csharp
// C#
interface IRepository<T>
{
    T? Get(int id);
    void Add(int id, T item);
}

class InMemoryRepository<T> : IRepository<T> { /* ... */ }
```

In [ ]:
from typing import Protocol


class Repository[T](Protocol):
    def get(self, item_id: int) -> T | None: ...
    def add(self, item_id: int, item: T) -> None: ...


class InMemoryRepository[T]:              # note: no inheritance from Repository
    def __init__(self) -> None:
        self._items: dict[int, T] = {}

    def get(self, item_id: int) -> T | None:
        return self._items.get(item_id)

    def add(self, item_id: int, item: T) -> None:
        self._items[item_id] = item


def seed(repo: Repository[str]) -> None:  # accepts anything with the right shape
    repo.add(1, "alice")
    repo.add(2, "bob")


users = InMemoryRepository[str]()
seed(users)
print(users.get(1), users.get(3))

## No runtime reification (type erasure)

In .NET, generics are **reified**: `List<int>` and `List<string>` are distinct runtime types, and you can use `typeof(T)` or `x is List<int>`. Python generics are **erased** — like Java. `Box[int]` is only meaningful to the type checker.

```csharp
// C#
var b = new Box<int>(1);
Console.WriteLine(b.GetType());          // Box`1[System.Int32]
Console.WriteLine(b is Box<string>);     // False
```

In [ ]:
b1 = Box[int](1)
b2 = Box[str]("one")
print(type(b1), type(b2), type(b1) is type(b2))   # both are just Box

wrong = Box[int]("not an int")                    # no runtime check at all
print(wrong)

for check in (lambda: isinstance(b1, Box[int]), lambda: isinstance([1], list[int])):
    try:
        check()
    except TypeError as e:
        print(f"{type(e).__name__}: {e}")

Consequences for C# developers:

- No `typeof(T)` inside a generic function — pass the type explicitly as a parameter (`cls: type[T]`) if you need it.
- No `new T()` — accept a factory (`Callable[[], T]`) or `type[T]` instead.
- Overloading by type argument is impossible at runtime.

In [ ]:
def create_all[T](cls: type[T], count: int) -> list[T]:   # ≈ where T : new()
    return [cls() for _ in range(count)]


print(create_all(list, 2), create_all(int, 3))

## Summary

| Concept | C# | Python |
|---|---|---|
| Generic function | `T First<T>(IList<T> xs)` | `def first[T](xs: Sequence[T]) -> T:` |
| Generic class | `class Box<T>` | `class Box[T]:` |
| Upper bound | `where T : IComparable<T>` | `[T: Comparable]` |
| Fixed set of types | — | `[T: (str, bytes)]` |
| Old syntax | — | `T = TypeVar("T")`, `class C(Generic[T])` |
| Covariance | `out T` (declared) | inferred (or `covariant=True`) |
| Contravariance | `in T` (declared) | inferred (or `contravariant=True`) |
| Generic interface | `interface IRepo<T>` | `class Repo[T](Protocol):` |
| `typeof(T)` | ✅ reified | ❌ erased — pass `type[T]` |
| `new T()` | `where T : new()` | pass `type[T]` or a factory |

## Exercises

1. Write a generic function `last[T](items: Sequence[T]) -> T | None` that returns the last item, or `None` for an empty sequence.
2. Write a generic class `Stack[T]` with `push(item: T)`, `pop() -> T` and `__len__`. Use `Stack[str]()` to push and pop a few values.
3. Write `clamp[T: Comparable](value: T, low: T, high: T) -> T` using only `<` comparisons. Test it with ints and with strings.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
from collections.abc import Sequence
from typing import Any, Protocol

# 1
def last[T](items: Sequence[T]) -> T | None:
    return items[-1] if items else None

print(last([1, 2, 3]), last([]))   # 3 None

# 2
class Stack[T]:
    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> T:
        return self._items.pop()

    def __len__(self) -> int:
        return len(self._items)

s = Stack[str]()
s.push("a"); s.push("b")
print(s.pop(), len(s))   # b 1

# 3
class Comparable(Protocol):
    def __lt__(self, other: Any, /) -> bool: ...

def clamp[T: Comparable](value: T, low: T, high: T) -> T:
    if value < low:
        return low
    if high < value:
        return high
    return value

print(clamp(15, 0, 10), clamp("m", "a", "f"))   # 10 f
```

</details>